# Refined pipeline quick smoke test

This notebook gives a tiny end-to-end **local** sanity check for the new API:
- `dense_grid_from_top_k` on synthetic coarse results + synthetic human runs
- minimal command examples for CLI usage


In [ ]:
import tempfile
from pathlib import Path
import numpy as np

from src.model.run_prior_guided_refined_pipeline import (
    DEFAULT_ISIS,
    dense_grid_from_top_k,
)

print('DEFAULT_ISIS =', DEFAULT_ISIS)

In [ ]:
# 1) Build a tiny synthetic coarse-grid result file
tmpdir = Path(tempfile.mkdtemp(prefix='refined_pipeline_demo_'))
coarse_path = tmpdir / 'grid_search_results_prior_guided.npz'

sigma0_grid = np.array([0.05, 0.10, 0.20], dtype=float)
sigma_grid  = np.array([0.01, 0.05, 0.10], dtype=float)
eta_grid    = np.array([0.001, 0.01, 0.10], dtype=float)
isi_values  = np.array([0, 1, 2, 4, 8, 16, 32, 64], dtype=int)

shape = (len(sigma0_grid), len(sigma_grid), len(eta_grid))
rng = np.random.default_rng(7)
dprime_by_isi = {f'dprime_isi{isi}': rng.normal(0.8, 0.15, size=shape) for isi in isi_values}

np.savez(
    coarse_path,
    sigma0_grid=sigma0_grid,
    sigma_grid=sigma_grid,
    eta_grid=eta_grid,
    isi_values=isi_values,
    **dprime_by_isi,
)

coarse_path

In [ ]:
# 2) Build toy human runs in the shape expected by roc_for_isi
# Each run needs: isi_hit_dists (dict: isi->[(score, t)]), fas (array), score_type
human_runs = []
rng = np.random.default_rng(123)

for _ in range(20):
    run = {
        'isi_hit_dists': {},
        'fas': rng.normal(1.1, 0.1, size=200),
        'score_type': 'distance',
    }
    for isi in isi_values:
        scores = rng.normal(0.8 - 0.01 * isi, 0.08, size=70)
        run['isi_hit_dists'][int(isi)] = [(float(s), int(20 + j)) for j, s in enumerate(scores)]
    human_runs.append(run)

len(human_runs)

In [ ]:
# 3) Run dense-grid proposal from robust top-K
dense = dense_grid_from_top_k(
    str(coarse_path),
    human_runs,
    model_type='prior',
    top_k=5,
    n_points=6,
    n_folds=6,
    seed=42,
)

print('keys:', sorted(dense.keys()))
print('sigma0_grid:', dense['sigma0_grid'])
print('sigma_grid :', dense['sigma_grid'])
print('eta_grid   :', dense['eta_grid'])
print('param_ranges:', dense['param_ranges'])
print('num robust top params:', len(dense['top_k_params']))

## Minimal CLI usage (real data)
```bash
# 1) Build dense grid spec from coarse merged file
python src/model/run_prior_guided_refined_pipeline.py \
  --model-type prior \
  --mode dense-grid \
  --coarse-results-path /path/to/grid_search_results_prior_guided.npz \
  --save-dir /tmp/refined_prior

# 2) Run one fine-search array point
python src/model/run_prior_guided_refined_pipeline.py \
  --model-type prior \
  --mode fine-search \
  --save-dir /tmp/refined_prior \
  --job-index 0 --parallel-mode flat --n-mc 2

# 3) Merge outputs
python src/model/run_prior_guided_refined_pipeline.py \
  --model-type prior --mode merge-fine --save-dir /tmp/refined_prior

# 4) ISI=16 transfer for top models
python src/model/run_prior_guided_refined_pipeline.py \
  --model-type prior --mode transfer-isi16 --save-dir /tmp/refined_prior --n-mc 2
```
